# 📈 Sistema de Trading Automatizado v5.2 – GEBRA Black Belt (Macro + Técnico + Estatístico)
**Novidades da v5.2:**
- **Teoria de Dow**: detecção de tendência primária.
- **Fibonacci (retrações)**: níveis de 38,2%, 50%, 61,8%.
- **Ondas de Elliott**: classificação simplificada da onda atual.
- **Indicadores Técnicos**: RSI, MACD, Estocástico, Bandas de Bollinger.
- **Sentimento de Mercado**: Clímax de volume, NH‑NL (saúde do mercado).
- Todas as informações disponíveis na tabela HTML e CSV.
Mantém v5.1 (padrões clássicos, candlesticks, zonas psicológicas, armadilhas).

In [ ]:
# =============================================================================
# CÉLULA 0: PARÂMETROS GLOBAIS + CONFIGURAÇÃO DE LOGGING
# =============================================================================
# ⚠️ SEGURANÇA: E-mail e senha são obtidos EXCLUSIVAMENTE via userdata do Colab.
# Nunca comite este notebook com valores reais.
EMAIL_REMETENTE = ""  # Será preenchido via userdata.get('TRADING_EMAIL')
SENHA_APP = ""       # Será preenchida via userdata.get('GMAIL_APP_PASSWORD')

PARAMS_BAIXA_VOL = {
    'kelly_frac': 0.30,
    'wyckoff_threshold': 0.75,
    'gap_max_pct': 0.055,
    'custos_pct': 0.003,
    'exigir_volume_anormal': False,
    'risco_percentual_maximo': 0.15,
    'preco_minimo': 2.00
}

PARAMS_ALTA_VOL = {
    'kelly_frac': 0.15,
    'wyckoff_threshold': 0.85,
    'gap_max_pct': 0.03,
    'custos_pct': 0.006,
    'exigir_volume_anormal': True,
    'risco_percentual_maximo': 0.10,
    'preco_minimo': 2.00
}

PARAMS_ATIVOS = PARAMS_BAIXA_VOL.copy()

MAX_SETUPS_POR_DIA = 5
MAX_PERDAS_CONSECUTIVAS = 3
DRAWDOWN_MAX_DIARIO = 0.02
MAX_DIAS_LOG = 30

HABILITAR_LOGGING = True
ARQUIVO_LOG = "trading_log_v52.json"
ARQUIVO_LOG_DETALHADO = "execucao_detalhada_v52.log"

CAPITAL_TOTAL = 100000.0
WIN_RATE_ESTIMADO = 0.40
PAYOFF_ESTIMADO = 3.0

SETORES_BLOQUEADOS = ['AEREA']
TICKERS_BLOQUEADOS = ['GFSA3.SA', 'ONCO3.SA', 'PMAM3.SA', 'AZTE3.SA', 'RAIZ4.SA',
                      'BHIA3.SA', 'CASH3.SA', 'LJQQ3.SA', 'RCSL4.SA', 'HBOR3.SA']

FALLBACK_TICKERS = [
    'PETR4', 'VALE3', 'ITUB4', 'BBDC4', 'BBAS3', 'ABEV3',
    'WEGE3', 'RADL3', 'SUZB3', 'GGBR4', 'MGLU3', 'VVAR3',
    'RENT3', 'RAIL3', 'CCRO3', 'ELET3', 'CPFE3', 'SBSP3',
    'SANB11', 'B3SA3', 'JBSS3', 'BRFS3', 'KLBN11', 'EQTL3'
]

RISCO_PERCENTUAL_MINIMO = 0.02
RISCO_PERCENTUAL_MAXIMO = 0.10
PRECO_MINIMO = 5.00

BANDA_ZONA_PCT = 0.01
EXIGIR_CONFLUENCIA_CANDLE = True
CACHE_TICKERS_FILE = "cache_tickers_b3.json"
CACHE_MACRO_EXPIRY_HORAS = 24

LOG_PERFORMANCE = True
LOG_FILTROS_DETALHADO = True

In [ ]:
# =============================================================================
# CÉLULA 1: INSTALAÇÃO E LOGGING
# =============================================================================
!pip install yfinance pandas-ta --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
import requests
from bs4 import BeautifulSoup
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime, timedelta
import time, warnings, json, os, sys
from typing import Optional, Tuple, Dict, List
import traceback

warnings.filterwarnings("ignore")

class Logger:
    def __init__(self, arquivo_log, arquivo_detalhado=None):
        self.arquivo_log = arquivo_log
        self.arquivo_detalhado = arquivo_detalhado
        self.inicio_geral = time.time()
        self.timings = {}
        self.contadores = {}
        self._buffer = []
    def log(self, mensagem, nivel="INFO", ticker=None):
        ts = datetime.now().strftime("%H:%M:%S")
        msg = f"[{ts}] [{nivel}] {mensagem}"
        if ticker: msg += f" | {ticker}"
        print(msg)
        if self.arquivo_detalhado and LOG_PERFORMANCE:
            self._buffer.append(msg + "\n")
            if len(self._buffer) >= 50: self._flush_buffer()
    def _flush_buffer(self):
        if self._buffer and self.arquivo_detalhado:
            with open(self.arquivo_detalhado, 'a', encoding='utf-8') as f: f.writelines(self._buffer)
            self._buffer.clear()
    def iniciar_etapa(self, nome):
        self.timings[nome] = {'inicio': time.time()}
        self.log(f"🚀 Iniciando: {nome}", "ETAPA")
    def concluir_etapa(self, nome, detalhes=None):
        if nome in self.timings:
            dur = time.time() - self.timings[nome]['inicio']
            self.timings[nome]['duracao'] = dur
            msg = f"✅ Concluído: {nome} ({dur:.2f}s)"
            if detalhes: msg += " | " + " | ".join(f"{k}: {v}" for k,v in detalhes.items())
            self.log(msg, "ETAPA")
    def incrementar(self, contador, valor=1):
        self.contadores[contador] = self.contadores.get(contador, 0) + valor
    def resumo_final(self):
        self._flush_buffer()
        total = time.time() - self.inicio_geral
        self.log("\n" + "="*60, "RESUMO")
        self.log(f"⏱️ Tempo total: {total:.2f}s", "RESUMO")
        for etapa, dados in self.timings.items():
            if 'duracao' in dados:
                self.log(f"   • {etapa}: {dados['duracao']:.2f}s ({dados['duracao']/total*100:.1f}%)", "RESUMO")
        if self.contadores:
            self.log("\n🔢 Contadores:", "RESUMO")
            for cont, val in self.contadores.items(): self.log(f"   • {cont}: {val}", "RESUMO")
        self.log("="*60 + "\n", "RESUMO")
        with open('resumo_execucao.json', 'w', encoding='utf-8') as f:
            json.dump({'timestamp': datetime.now().isoformat(), 'duracao_total_segundos': total, 'timings': {k: {kk: vv for kk, vv in v.items() if kk != 'inicio'} for k, v in self.timings.items()}, 'contadores': self.contadores}, f, indent=2, ensure_ascii=False)

logger = Logger(ARQUIVO_LOG, ARQUIVO_LOG_DETALHADO)

def log_evento(tipo, ticker, dados, arquivo=ARQUIVO_LOG, max_dias=MAX_DIAS_LOG):
    if not HABILITAR_LOGGING: return
    registro = {'timestamp': datetime.now().isoformat(), 'tipo': tipo, 'ticker': ticker, 'dados': dados}
    logs = []
    if os.path.exists(arquivo):
        try:
            with open(arquivo, 'r', encoding='utf-8') as f: logs = json.load(f)
        except: logs = []
    cutoff = datetime.now() - timedelta(days=max_dias)
    logs = [l for l in logs if datetime.fromisoformat(l['timestamp']) > cutoff]
    logs.append(registro)
    with open(arquivo, 'w', encoding='utf-8') as f: json.dump(logs, f, ensure_ascii=False, indent=2)

logger.log("🔧 Sistema de logging inicializado", "INFO")
print("✅ Célula 1 carregada.")

In [ ]:
# =============================================================================
# CÉLULA 2: FUNÇÕES AUXILIARES (v5.2 – Macro + Técnico + Estatístico)
# =============================================================================

def calcular_eficiencia_candle(df):
    corpo = abs(df['Close'] - df['Open'])
    sombra_sup = df['High'] - df[['Close', 'Open']].max(axis=1)
    sombra_inf = df[['Close', 'Open']].min(axis=1) - df['Low']
    range_total = df['High'] - df['Low']
    range_total = range_total.replace(0, np.nan)
    eficiencia = pd.Series(index=df.index, dtype=float)
    alta, baixa = df['Close'] > df['Open'], df['Close'] < df['Open']
    eficiencia[alta] = 1 - (sombra_sup[alta] / range_total[alta])
    eficiencia[baixa] = 1 - (sombra_inf[baixa] / range_total[baixa])
    return eficiencia

def detectar_regime(df, janela=20):
    df_temp = df.copy()
    df_temp['retorno'] = df_temp['Close'].pct_change()
    df_temp['volatilidade'] = df_temp['retorno'].rolling(janela).std()
    try:
        adx = ta.adx(df_temp['High'], df_temp['Low'], df_temp['Close'], length=14)
        if adx is None or (isinstance(adx, pd.DataFrame) and 'ADX_14' not in adx.columns):
            if adx is not None:
                df_temp['adx'] = adx[adx.columns[0]]
            else:
                logger.log("ADX None em detectar_regime", "WARN")
                return pd.Series(index=df.index, dtype=int)
        else:
            df_temp['adx'] = adx['ADX_14']
    except Exception as e:
        logger.log(f"Erro ADX: {e}", "WARN")
        return pd.Series(index=df.index, dtype=int)
    df_temp = df_temp.dropna(subset=['volatilidade', 'adx'])
    if df_temp.empty: return pd.Series(index=df.index, dtype=int)
    v_p33, v_p67 = df_temp['volatilidade'].quantile([0.33, 0.67])
    a_p33, a_p67 = df_temp['adx'].quantile([0.33, 0.67])
    def classificar(row):
        v, a = row['volatilidade'], row['adx']
        if v < v_p33 and a < a_p33: return 0
        elif v > v_p67 or a > a_p67: return 2
        else: return 1
    regimes = df_temp.apply(classificar, axis=1)
    regime_series = pd.Series(index=df.index, dtype=int)
    regime_series.loc[regimes.index] = regimes
    regime_series.ffill(inplace=True)
    return regime_series

def detectar_swing_low(df, janela=10, confirmar=True):
    lows, closes = df['Low'].values, df['Close'].values
    swing_lows = []
    for i in range(janela, len(lows)):
        if lows[i] <= min(lows[i-janela:i]):
            if not confirmar or (i+1 < len(lows) and closes[i+1] > lows[i]):
                swing_lows.append(lows[i])
    return float(swing_lows[-1]) if swing_lows else float(df['Low'].min())

def detectar_swing_high(df, janela=10, confirmar=True):
    highs, closes = df['High'].values, df['Close'].values
    swing_highs = []
    for i in range(janela, len(highs)):
        if highs[i] >= max(highs[i-janela:i]):
            if not confirmar or (i+1 < len(highs) and closes[i+1] < highs[i]):
                swing_highs.append(highs[i])
    return float(swing_highs[-1]) if swing_highs else float(df['High'].max())

def calcular_lta_pivos(df, janela_pivo=5):
    lows = df['Low'].values
    lows_seguro = np.where((lows > 0) & np.isfinite(lows), lows, np.nan)
    if np.isnan(lows_seguro).all(): return None
    log_lows = np.log(lows_seguro)
    fundos = []
    for i in range(janela_pivo, len(log_lows) - janela_pivo):
        if np.isnan(log_lows[i]): continue
        if log_lows[i] == min(log_lows[i-janela_pivo:i+janela_pivo+1]):
            fundos.append((i, log_lows[i]))
    if len(fundos) >= 2:
        (x1, y1), (x2, y2) = fundos[-2], fundos[-1]
        if x2 == x1: return None
        incl = (y2 - y1) / (x2 - x1)
        return np.exp(y2 + incl * (len(log_lows) - 1 - x2))
    return None

def calcular_ltb_pivos(df, janela_pivo=5):
    highs = df['High'].values
    highs_seguro = np.where((highs > 0) & np.isfinite(highs), highs, np.nan)
    if np.isnan(highs_seguro).all(): return None
    log_highs = np.log(highs_seguro)
    topos = []
    for i in range(janela_pivo, len(log_highs) - janela_pivo):
        if np.isnan(log_highs[i]): continue
        if log_highs[i] == max(log_highs[i-janela_pivo:i+janela_pivo+1]):
            topos.append((i, log_highs[i]))
    if len(topos) >= 2 and topos[-2][1] > topos[-1][1]:
        (x1, y1), (x2, y2) = topos[-2], topos[-1]
        if x2 == x1: return None
        incl = (y2 - y1) / (x2 - x1)
        return np.exp(y2 + incl * (len(log_highs) - 1 - x2))
    return None

def detectar_armadilha_lta(df, lta, banda_pct=0.01):
    if len(df) < 2: return False, 0.0
    ult = df.iloc[-1]
    low, close, open_ = ult['Low'], ult['Close'], ult['Open']
    corpo = abs(close - open_)
    range_c = ult['High'] - low
    if range_c == 0: return False, 0.0
    sombra_inf = min(close, open_) - low
    zona_inf = lta * (1 - banda_pct)
    if low < zona_inf and close >= zona_inf and sombra_inf >= 2 * corpo:
        return True, min(1.0, sombra_inf / range_c)
    return False, 0.0

def detectar_armadilha_ltb(df, ltb, banda_pct=0.01):
    if len(df) < 2: return False, 0.0
    ult = df.iloc[-1]
    high, close, open_ = ult['High'], ult['Close'], ult['Open']
    corpo = abs(close - open_)
    range_c = high - ult['Low']
    if range_c == 0: return False, 0.0
    sombra_sup = high - max(close, open_)
    zona_sup = ltb * (1 + banda_pct)
    if high > zona_sup and close <= zona_sup and sombra_sup >= 2 * corpo:
        return True, min(1.0, sombra_sup / range_c)
    return False, 0.0

def calcular_lta_adaptativo(df, janelas=None):
    if janelas is None: janelas = [20, 30, 40, 50]
    if len(df) < max(janelas): return None
    lows = df['Low'].values
    closes = df['Close'].values
    melhor_score = -np.inf
    melhor = None
    for janela in janelas:
        lta = calcular_lta_pivos(df, janela_pivo=janela)
        if lta is None: continue
        zona_inf = lta * (1 - BANDA_ZONA_PCT)
        zona_sup = lta * (1 + BANDA_ZONA_PCT)
        ini = max(0, len(lows) - max(2*janela, 50))
        suportes = np.sum(lows[ini:] >= zona_inf)
        violacoes = np.sum(closes[ini:] < zona_inf)
        toques_zona = np.sum((lows[ini:] >= zona_inf) & (lows[ini:] <= zona_sup))
        score = suportes - 3 * violacoes + 2 * toques_zona
        if score > melhor_score:
            melhor_score = score
            melhor = (lta, janela)
    return melhor

def calcular_ltb_adaptativo(df, janelas=None):
    if janelas is None: janelas = [20, 30, 40, 50]
    if len(df) < max(janelas): return None
    highs = df['High'].values
    closes = df['Close'].values
    melhor_score = -np.inf
    melhor = None
    for janela in janelas:
        ltb = calcular_ltb_pivos(df, janela_pivo=janela)
        if ltb is None: continue
        zona_inf = ltb * (1 - BANDA_ZONA_PCT)
        zona_sup = ltb * (1 + BANDA_ZONA_PCT)
        ini = max(0, len(highs) - max(2*janela, 50))
        resistencias = np.sum(highs[ini:] <= zona_sup)
        violacoes = np.sum(closes[ini:] > zona_sup)
        toques_zona = np.sum((highs[ini:] >= zona_inf) & (highs[ini:] <= zona_sup))
        score = resistencias - 3 * violacoes + 2 * toques_zona
        if score > melhor_score:
            melhor_score = score
            melhor = (ltb, janela)
    return melhor

# ===== PADRÕES CLÁSSICOS (v5.1) =====
def detectar_oco(df, janela_swing=5):
    if len(df) < 50: return None
    highs, lows = df['High'].values, df['Low'].values
    swings_high, swings_low = [], []
    for i in range(janela_swing, len(highs)-janela_swing):
        if highs[i] >= max(highs[i-janela_swing:i+janela_swing+1]):
            swings_high.append((i, highs[i]))
        if lows[i] <= min(lows[i-janela_swing:i+janela_swing+1]):
            swings_low.append((i, lows[i]))
    if len(swings_high) >= 3:
        s1, h, s2 = swings_high[-3], swings_high[-2], swings_high[-1]
        if s1[1] < h[1] and s2[1] < h[1] and abs(s1[1] - s2[1]) / max(s1[1], s2[1]) < 0.1:
            return {'tipo': 'OCO'}
    if len(swings_low) >= 3:
        v1, vv, v2 = swings_low[-3], swings_low[-2], swings_low[-1]
        if v1[1] > vv[1] and v2[1] > vv[1] and abs(v1[1] - v2[1]) / max(v1[1], v2[1]) < 0.1:
            return {'tipo': 'OCO_invertido'}
    return None

def detectar_bandeira(df):
    if len(df) < 10: return None
    closes = df['Close'].values
    variacao = closes[-1] - closes[0]
    if abs(variacao) < 0.05 * closes[0]: return None
    direcao = 'ALTA' if variacao > 0 else 'BAIXA'
    highs_cons = df['High'].iloc[-6:-1]
    if (highs_cons.max() - highs_cons.min()) < 0.02 * closes[-1]:
        return {'tipo': f'Bandeira_{direcao}'}
    return None

def detectar_triangulo_ascendente(df, janela=20):
    if len(df) < janela: return None
    resistencias = df['High'].rolling(janela).max().iloc[-janela:-1]
    suportes = df['Low'].rolling(janela).min().iloc[-janela:-1]
    if pd.Series(resistencias).std() < 0.01 * resistencias.mean():
        if pd.Series(suportes).is_monotonic_increasing:
            return {'tipo': 'Triangulo_asc'}
    return None

def detectar_triangulo_descendente(df, janela=20):
    if len(df) < janela: return None
    suportes = df['Low'].rolling(janela).min().iloc[-janela:-1]
    resistencias = df['High'].rolling(janela).max().iloc[-janela:-1]
    if pd.Series(suportes).std() < 0.01 * suportes.mean():
        if pd.Series(resistencias).is_monotonic_decreasing:
            return {'tipo': 'Triangulo_desc'}
    return None

def classificar_gap(df, direcao):
    if len(df) < 3: return None
    hoje, ontem = df.iloc[-1], df.iloc[-2]
    gap = (hoje['Open'] - ontem['Close']) / ontem['Close']
    if abs(gap) > 0.02:
        if direcao == 'COMPRA' and gap > 0:
            if hoje['Close'] > hoje['Open']: return {'tipo': 'Gap_fuga_alta'}
            else: return {'tipo': 'Gap_exaustao'}
        elif direcao == 'VENDA' and gap < 0:
            if hoje['Close'] < hoje['Open']: return {'tipo': 'Gap_fuga_baixa'}
            else: return {'tipo': 'Gap_exaustao'}
    return None

# ===== CANDLESTICKS =====
def analisar_candle(row):
    open_, high, low, close = row['Open'], row['High'], row['Low'], row['Close']
    corpo = abs(close - open_)
    range_total = high - low
    if range_total == 0: return {}
    p_sup = high - max(open_, close)
    p_inf = min(open_, close) - low
    res = {}
    if p_inf >= 2*corpo and p_sup <= 0.3*range_total and corpo > 0:
        res['martelo'] = True
    if p_sup >= 2*corpo and p_inf <= 0.3*range_total and corpo > 0:
        res['estrela_cadente'] = True
    if corpo <= 0.05 * range_total:
        res['doji'] = True
    return res

# ===== MACROMOVIMENTOS (v5.2) =====
def detectar_estrutura_dow(df):
    if len(df) < 26: return None
    ult_top, prev_top = df['High'].iloc[-1], df['High'].iloc[-26]
    ult_fnd, prev_fnd = df['Low'].iloc[-1], df['Low'].iloc[-26]
    if ult_top > prev_top and ult_fnd > prev_fnd:
        return {'tendencia_dow': 'ALTA'}
    elif ult_top < prev_top and ult_fnd < prev_fnd:
        return {'tendencia_dow': 'BAIXA'}
    else:
        return {'tendencia_dow': 'LATERAL'}

def calcular_fibonacci_retracao(df):
    if len(df) < 50: return None
    sh = detectar_swing_high(df, janela=10, confirmar=False)
    sl = detectar_swing_low(df, janela=10, confirmar=False)
    if sh is None or sl is None or sh <= sl: return None
    diff = sh - sl
    return {'38.2%': round(sh - diff * 0.382, 2),
            '50.0%': round(sh - diff * 0.500, 2),
            '61.8%': round(sh - diff * 0.618, 2)}

def classificar_onda_elliott(df):
    if len(df) < 20: return None
    variacao = (df['Close'].iloc[-1] - df['Close'].iloc[-20]) / df['Close'].iloc[-20]
    if abs(variacao) < 0.05: return None
    if abs(variacao) > 0.15:
        return 'Possível Onda 3 (impulso forte)' if variacao > 0 else 'Possível Onda 3 (queda forte)'
    else:
        return 'Possível Onda 5 (exaustão)' if variacao > 0 else 'Possível Onda 5 (exaustão vendedora)'

# ===== INDICADORES TÉCNICOS (v5.2) =====
def calcular_rsi(df, periodo=14):
    try:
        rsi = ta.rsi(df['Close'], length=periodo)
        return round(float(rsi.iloc[-1]), 1) if not rsi.empty else None
    except: return None

def calcular_macd(df):
    try:
        macd = ta.macd(df['Close'])
        if macd is not None and not macd.empty:
            return (round(float(macd['MACD_12_26_9'].iloc[-1]), 2),
                    round(float(macd['MACDs_12_26_9'].iloc[-1]), 2),
                    round(float(macd['MACDh_12_26_9'].iloc[-1]), 2))
    except: pass
    return None, None, None

def calcular_estocastico(df, periodo=14):
    try:
        stoch = ta.stoch(df['High'], df['Low'], df['Close'], k=periodo, d=3)
        if stoch is not None and not stoch.empty:
            return (round(float(stoch['STOCHk_14_3_3'].iloc[-1]), 1),
                    round(float(stoch['STOCHd_14_3_3'].iloc[-1]), 1))
    except: pass
    return None, None

def calcular_bandas_bollinger(df, periodo=20):
    try:
        bb = ta.bbands(df['Close'], length=periodo)
        if bb is not None and not bb.empty:
            upper = float(bb['BBU_20_2.0'].iloc[-1])
            mid = float(bb['BBM_20_2.0'].iloc[-1])
            lower = float(bb['BBL_20_2.0'].iloc[-1])
            close = float(df['Close'].iloc[-1])
            pos = (close - lower) / (upper - lower) * 100 if upper != lower else 50
            return {'upper': round(upper,2), 'mid': round(mid,2), 'lower': round(lower,2), 'posicao_%': round(pos,1)}
    except: pass
    return None

def calcular_climax_volume(df, periodo=50):
    if len(df) < periodo: return False
    vol_med = df['Volume'].rolling(periodo).mean().iloc[-1]
    vol_at = df['Volume'].iloc[-1]
    return vol_at > 3 * vol_med if pd.notna(vol_med) else False

def calcular_nh_nl_simplificado(tickers_liquidos, data_w):
    try:
        count, total = 0, 0
        for t in tickers_liquidos:
            if t not in data_w or data_w[t] is None or data_w[t].empty: continue
            try:
                mm50 = data_w[t]['Close'].rolling(50).mean().iloc[-1]
                close = data_w[t]['Close'].iloc[-1]
                if pd.notna(mm50) and close > mm50: count += 1
                total += 1
            except: continue
        return round(count / total * 100, 1) if total > 0 else None
    except: return None

# ===== FUNÇÕES ORIGINAIS MANTIDAS =====
def detectar_padrao_altista(df_diario):
    if len(df_diario) < 3: return False
    u, p = df_diario.iloc[-1], df_diario.iloc[-2]
    c_u = abs(u['Close'] - u['Open'])
    r_u = u['High'] - u['Low']
    if r_u > 0:
        si_u = min(u['Close'], u['Open']) - u['Low']
        ss_u = u['High'] - max(u['Close'], u['Open'])
        if si_u >= 2 * c_u and ss_u <= 0.3 * c_u: return True
    if p['Close'] < p['Open'] and u['Close'] > u['Open']:
        if u['Open'] <= p['Close'] and u['Close'] >= p['Open']: return True
        meio = (p['Open'] + p['Close']) / 2
        if u['Open'] <= p['Close'] and u['Close'] >= meio: return True
    return False

def detectar_padrao_baixista(df_diario):
    if len(df_diario) < 3: return False
    u, p = df_diario.iloc[-1], df_diario.iloc[-2]
    c_u = abs(u['Close'] - u['Open'])
    r_u = u['High'] - u['Low']
    if r_u > 0:
        si_u = min(u['Close'], u['Open']) - u['Low']
        ss_u = u['High'] - max(u['Close'], u['Open'])
        if ss_u >= 2 * c_u and si_u <= 0.3 * c_u: return True
    if p['Close'] > p['Open'] and u['Close'] < u['Open']:
        if u['Open'] >= p['Close'] and u['Close'] <= p['Open']: return True
        meio = (p['Open'] + p['Close']) / 2
        if u['Open'] >= p['Close'] and u['Close'] <= meio: return True
    return False

SENTIMENTO_PADRAO = 0.0

def detectar_volume_anormal(df, periodo=20, limiar=1.5):
    if len(df) < periodo: return False
    vol_medio = df['Volume'].rolling(periodo).mean().iloc[-1]
    return df['Volume'].iloc[-1] >= vol_medio * limiar if pd.notna(vol_medio) else False

def fractional_kelly(win_rate, payoff_ratio, frac=0.25):
    if payoff_ratio <= 0: return 0.0
    kelly = (payoff_ratio * win_rate - (1 - win_rate)) / payoff_ratio
    return max(0.0, min(kelly, 0.25)) * frac

MACRO_REFERENCE = {
    'VALE3.SA': ('GC=F', 0.6), 'PETR4.SA': ('CL=F', 0.8), 'PETR3.SA': ('CL=F', 0.8),
    'CSNA3.SA': ('GC=F', 0.5), 'GGBR4.SA': ('GC=F', 0.5), 'CAML3.SA': ('WEAT', 0.4),
    'JBSS3.SA': ('WEAT', 0.5), 'ABEV3.SA': ('CORN', 0.3), 'RADL3.SA': ('XLP', 0.3),
    'PRIO3.SA': ('CL=F', 0.7),
}

_cache_macro_data = {}
_cache_macro_timestamp = {}

def _obter_dados_macro(bench):
    agora = datetime.now()
    if bench in _cache_macro_timestamp:
        idade = (agora - _cache_macro_timestamp[bench]).total_seconds() / 3600
        if idade < CACHE_MACRO_EXPIRY_HORAS:
            return _cache_macro_data[bench]
    try:
        df_b = yf.download(bench, period='1y', interval='1wk', progress=False, auto_adjust=True)
        vals = df_b['Close'].values if not df_b.empty else None
        _cache_macro_data[bench] = vals
        _cache_macro_timestamp[bench] = agora
        return vals
    except:
        return None

def verificar_alinhamento_macro(ticker, direcao, _=None):
    if ticker not in MACRO_REFERENCE: return True, 15
    ref, _ = MACRO_REFERENCE[ticker]
    precos = _obter_dados_macro(ref)
    if precos is None or len(precos) < 55: return True, 15
    ema50 = pd.Series(precos).ewm(span=50, adjust=False).mean()
    e_at, e_lag = ema50.iloc[-1], ema50.iloc[-5]
    if pd.isna(e_at) or pd.isna(e_lag): return True, 15
    slope_pos = e_at > e_lag
    if direcao == 'COMPRA' and precos[-1] > e_at and slope_pos: return True, 30
    if direcao == 'VENDA' and precos[-1] < e_at and not slope_pos: return True, 30
    return False, 0

def avaliar_qualidade_volume(df_w, direcao):
    vol_u = df_w['Volume'].iloc[-1]
    vol_m = df_w['Volume'].rolling(20).mean().iloc[-1]
    ef = df_w['Eficiencia'].iloc[-1] if 'Eficiencia' in df_w.columns else 0.5
    if pd.isna(vol_m) or vol_m == 0: return 'NEUTRO', 10
    if vol_u > vol_m * 1.5:
        if direcao == 'COMPRA' and ef > 0.7: return 'ALTA_CONVICCAO', 25
        if direcao == 'VENDA' and ef > 0.7: return 'ALTA_CONVICCAO', 25
        if ef < 0.5: return 'POSSIVEL_ARMADILHA', 15
    return 'NEUTRO', 10

def detectar_fase_wyckoff_adaptativo(df_w, suporte=None, resistencia=None, atr_period=14):
    if len(df_w) < 30: return 'INDEFINIDO', 0.0
    atr = ta.atr(df_w['High'], df_w['Low'], df_w['Close'], length=atr_period)
    if atr is None or atr.empty: return 'INDEFINIDO', 0.0
    atr_med = atr.rolling(20).mean()
    atr_val, atr_med_val = atr.iloc[-1], atr_med.iloc[-1]
    vol_rel = atr_val / atr_med_val if (pd.notna(atr_med_val) and atr_med_val != 0) else 1.0
    thr_base = PARAMS_ATIVOS.get('wyckoff_threshold', 0.75)
    thr_comp = max(0.65, min(0.90, thr_base + 0.10 * (vol_rel - 1)))
    range_s = df_w['High'] - df_w['Low']
    r_med = range_s.rolling(20).mean().iloc[-1]
    if range_s.iloc[-1] >= r_med * thr_comp: return 'INDEFINIDO', 0.0
    px = df_w['Close'].iloc[-1]
    if suporte is None: suporte = df_w['Low'].rolling(20).min().iloc[-1]
    if resistencia is None: resistencia = df_w['High'].rolling(20).max().iloc[-1]
    faixa = resistencia - suporte
    if faixa == 0: return 'INDEFINIDO', 0.0
    pos_rel = (px - suporte) / faixa
    candles = df_w.iloc[-8:]
    alta = candles['Close'] > candles['Open']
    baixa = candles['Close'] < candles['Open']
    vol_alta = candles.loc[alta, 'Volume'].mean() if alta.any() else 0
    vol_baixa = candles.loc[baixa, 'Volume'].mean() if baixa.any() else 0
    if vol_baixa == 0: return 'INDEFINIDO', 0.0
    razao = vol_alta / vol_baixa
    conf = min(1.0, (r_med - range_s.iloc[-1]) / r_med) if r_med > 0 else 0.5
    if pos_rel < 0.4 and razao > 1.2: return 'ACUMULACAO', conf
    if pos_rel > 0.6 and razao < 0.8: return 'DISTRIBUICAO', conf
    return 'INDEFINIDO', 0.0

def calcular_alvos_fibonacci(df, direcao, fib_window=20):
    if len(df) < fib_window: return {}
    try:
        df_rec = df.iloc[-fib_window:]
        sl = detectar_swing_low(df_rec, janela=5, confirmar=False)
        sh = detectar_swing_high(df_rec, janela=5, confirmar=False)
        if sl is None or sh is None or sl >= sh: return {}
        amp = np.log(sh) - np.log(sl)
        base = np.log(sh)
        mults = [1.000, 1.618, 2.618, 4.236]
        if direcao == 'COMPRA':
            return {f'{m*100}%': round(np.exp(base + amp * m), 2) for m in mults}
        return {f'{m*100}%': round(np.exp(base - amp * m), 2) for m in mults}
    except:
        return {}

def calcular_score_confianca(setup, alinhado_macro, qualidade_volume, score_volume, fase_wyckoff, wyckoff_conf=1.0, is_armadilha=False, forca_armadilha=0.0):
    score = 0
    ef = setup.get('Eficiência')
    if ef and ef > 0.9: score += 40
    elif ef and ef > 0.8: score += 30
    elif ef and ef > 0.6: score += 15
    elif ef and ef >= 0.4: score += 5
    if is_armadilha: score += int(20 * forca_armadilha)
    if alinhado_macro: score += 20
    score += score_volume
    regime = setup.get('Regime')
    if regime == 2: score += 10
    elif regime == 1: score += 5
    if setup['Direcao'] == 'COMPRA' and fase_wyckoff == 'ACUMULACAO': score += int(20 * wyckoff_conf)
    elif setup['Direcao'] == 'VENDA' and fase_wyckoff == 'DISTRIBUICAO': score += int(20 * wyckoff_conf)
    return min(score, 100) / 100.0

def calcular_alvo_recomendado(setup, margem=0.03):
    fibos = setup.get('Alvos Fibonacci', {})
    if not fibos or '161.8%' not in fibos: return setup['Alvo 3:1'], '3:1'
    fib = fibos['161.8%']
    if setup['Direcao'] == 'COMPRA' and fib <= setup['Resistência'] * (1 + margem): return fib, 'Fib 161.8%'
    if setup['Direcao'] == 'VENDA' and fib >= setup['Suporte'] * (1 - margem): return fib, 'Fib 161.8%'
    return setup['Alvo 3:1'], '3:1'

def calcular_payoff_real(entrada, alvo, stop, custos_pct):
    risco = abs(entrada - stop)
    if risco == 0: return 0.0
    retorno_bruto = abs(alvo - entrada)
    custo_total = (entrada + alvo) * custos_pct
    retorno_liquido = max(0, retorno_bruto - custo_total)
    return round(retorno_liquido / risco, 2)

def detectar_regime_volatilidade(serie, janela=40):
    try:
        if serie is None or serie.empty or len(serie) < janela: return 'BAIXA'
        ret = serie.pct_change().dropna()
        if ret.empty or len(ret) < 20: return 'BAIXA'
        vol_at = ret.rolling(20).std().iloc[-1]
        vol_hist = ret.rolling(janela).std().dropna()
        if pd.isna(vol_at) or vol_hist.empty: return 'BAIXA'
        return 'ALTA' if (vol_hist < vol_at).mean() > 0.7 else 'BAIXA'
    except Exception:
        return 'BAIXA'

logger.log("✅ Funções auxiliares v5.2 carregadas", "INFO")
print("✅ Célula 2 carregada.")

In [ ]:
# =============================================================================
# CÉLULA 3: FUNÇÕES DE ANÁLISE (inalteradas)
# =============================================================================

OTIMIZADO_SWING = {'atr_period': 14, 'atr_mult': 1.8, 'swing_window': 12, 'lta_pivo_window': 6, 'ltb_pivo_window': 6, 'mm200_semanal': True, 'mm200_diaria': True}
OTIMIZADO_POSITION = {'atr_period': 14, 'atr_mult': 2.5, 'swing_window': 24, 'mm50_mensal': True}

def _normalizar_dataframe(df):
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.droplevel(1)
    rename_map = {'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}
    df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns}, inplace=True)
    df.sort_index(inplace=True)
    if not isinstance(df.index, pd.DatetimeIndex): df.index = pd.to_datetime(df.index)
    return df

def _safe_atr(df_high, df_low, df_close, length):
    try:
        atr_series = ta.atr(df_high, df_low, df_close, length=length)
        if atr_series is None or atr_series.empty: return 0.0
        val = atr_series.iloc[-1]
        return float(val) if pd.notna(val) else 0.0
    except: return 0.0

def analisar_swing_trade(ticker, df_w=None, df_d=None):
    try:
        if df_w is None: return None
        df_w = _normalizar_dataframe(df_w)
        df_w['Eficiencia'] = calcular_eficiencia_candle(df_w)
        df_w['Regime'] = detectar_regime(df_w)
        ult = df_w.iloc[-1]
        entrada = float(ult['Close'])
        if pd.isna(entrada) or entrada <= 0: return None
        rh = float(df_w['High'].rolling(window=min(52, len(df_w))).max().iloc[-1])
        rl = float(df_w['Low'].rolling(window=min(52, len(df_w))).min().iloc[-1])
        reg = int(ult['Regime']) if not pd.isna(ult['Regime']) else -1
        ef = round(float(ult['Eficiencia']), 2) if not pd.isna(ult['Eficiencia']) else None
        atr = _safe_atr(df_w['High'], df_w['Low'], df_w['Close'], OTIMIZADO_SWING['atr_period'])
        pa = pb = True
        if df_d is not None and not df_d.empty:
            df_loc = _normalizar_dataframe(df_d)
            pa = detectar_padrao_altista(df_loc)
            pb = detectar_padrao_baixista(df_loc)
        sentimento = SENTIMENTO_PADRAO
        vol_an = detectar_volume_anormal(df_w, periodo=20, limiar=1.5)
        setups = []
        if pa:
            fib = calcular_alvos_fibonacci(df_w, 'COMPRA')
            st_atr = entrada - (OTIMIZADO_SWING['atr_mult'] * atr) if atr > 0 else None
            st_sw = detectar_swing_low(df_w, janela=OTIMIZADO_SWING['swing_window'])
            st_sw = st_sw if st_sw and st_sw < entrada else None
            st_lta = calcular_lta_pivos(df_w, janela_pivo=OTIMIZADO_SWING['lta_pivo_window'])
            st_lta = st_lta if st_lta and st_lta < entrada else None
            for met, sl in [('ATR', st_atr), ('Swing Low', st_sw), ('LTA Pivôs', st_lta)]:
                if sl is None or sl <= 0 or sl >= entrada: continue
                r = entrada - sl
                al = entrada + (r * 3)
                if al <= rh * 1.05:
                    setups.append({'Ticker': ticker, 'Modalidade': 'Swing', 'Direcao': 'COMPRA', 'Entrada': round(entrada,2), 'Método Stop': met, 'Stop Loss': round(sl,2), 'Risco (R$)': round(r,2), 'Alvo 3:1': round(al,2), 'Resistência': round(rh,2), 'Suporte': round(rl,2), 'Regime': reg, 'Eficiência': ef, 'Sentimento': sentimento, 'Volume Anormal': vol_an, 'Alvos Fibonacci': fib})
        if pb:
            fib = calcular_alvos_fibonacci(df_w, 'VENDA')
            st_atr = entrada + (OTIMIZADO_SWING['atr_mult'] * atr) if atr > 0 else None
            st_sw = detectar_swing_high(df_w, janela=OTIMIZADO_SWING['swing_window'])
            st_sw = st_sw if st_sw and st_sw > entrada else None
            st_ltb = calcular_ltb_pivos(df_w, janela_pivo=OTIMIZADO_SWING['ltb_pivo_window'])
            st_ltb = st_ltb if st_ltb and st_ltb > entrada else None
            for met, sl in [('ATR', st_atr), ('Swing High', st_sw), ('LTB Pivôs', st_ltb)]:
                if sl is None or sl <= entrada: continue
                r = sl - entrada
                al = entrada - (r * 3)
                if al >= rl * 0.95:
                    setups.append({'Ticker': ticker, 'Modalidade': 'Swing', 'Direcao': 'VENDA', 'Entrada': round(entrada,2), 'Método Stop': met, 'Stop Loss': round(sl,2), 'Risco (R$)': round(r,2), 'Alvo 3:1': round(al,2), 'Resistência': round(rh,2), 'Suporte': round(rl,2), 'Regime': reg, 'Eficiência': ef, 'Sentimento': sentimento, 'Volume Anormal': vol_an, 'Alvos Fibonacci': fib})
        return setups if setups else None
    except Exception as e:
        if LOG_FILTROS_DETALHADO: logger.log(f"Erro swing {ticker}: {str(e)[:100]}", "ERRO")
        return None

def analisar_position_trade(ticker, df_m=None, df_w=None):
    try:
        if df_m is None or len(df_m) < 12: return None
        df_m = _normalizar_dataframe(df_m)
        df_m['Eficiencia'] = calcular_eficiencia_candle(df_m)
        df_m['Regime'] = detectar_regime(df_m)
        ult = df_m.iloc[-1]
        entrada = float(ult['Close'])
        if pd.isna(entrada) or entrada <= 0: return None
        mm12 = df_m['Close'].rolling(12).mean().iloc[-1]
        tendencia = 'ALTA' if entrada > mm12 else 'BAIXA'
        lb = min(60, len(df_m))
        rh = float(df_m['High'].rolling(window=lb).max().iloc[-1])
        rl = float(df_m['Low'].rolling(window=lb).min().iloc[-1])
        reg = int(ult['Regime']) if not pd.isna(ult['Regime']) else -1
        ef = round(float(ult['Eficiencia']), 2) if not pd.isna(ult['Eficiencia']) else None
        atr = _safe_atr(df_m['High'], df_m['Low'], df_m['Close'], OTIMIZADO_POSITION['atr_period'])
        sentimento = SENTIMENTO_PADRAO
        vol_an = detectar_volume_anormal(df_m, periodo=20, limiar=1.5)
        setups = []
        if tendencia == 'ALTA':
            fib_c = calcular_alvos_fibonacci(df_w, 'COMPRA') if df_w is not None else {}
            st_atr = entrada - (OTIMIZADO_POSITION['atr_mult'] * atr) if atr > 0 else None
            st_sw = detectar_swing_low(df_m, janela=OTIMIZADO_POSITION['swing_window'])
            st_sw = st_sw if st_sw and st_sw < entrada else None
            res_lta = calcular_lta_adaptativo(df_m)
            st_lta = res_lta[0] if res_lta else None
            if res_lta and LOG_FILTROS_DETALHADO: logger.log(f"📐 {ticker} LTA: janela={res_lta[1]}", "DEBUG")
            for met, sl in [('ATR', st_atr), ('Swing Low', st_sw), ('LTA Pivôs', st_lta)]:
                if sl is None or sl <= 0 or sl >= entrada: continue
                r = entrada - sl
                al = entrada + (r * 3)
                if al <= rh * 1.10:
                    setups.append({'Ticker': ticker, 'Modalidade': 'Position', 'Direcao': 'COMPRA', 'Entrada': round(entrada,2), 'Método Stop': met, 'Stop Loss': round(sl,2), 'Risco (R$)': round(r,2), 'Alvo 3:1': round(al,2), 'Resistência': round(rh,2), 'Suporte': round(rl,2), 'Regime': reg, 'Eficiência': ef, 'Sentimento': sentimento, 'Volume Anormal': vol_an, 'Alvos Fibonacci': fib_c})
        if tendencia == 'BAIXA':
            fib_v = calcular_alvos_fibonacci(df_w, 'VENDA') if df_w is not None else {}
            st_atr = entrada + (OTIMIZADO_POSITION['atr_mult'] * atr) if atr > 0 else None
            st_sw = detectar_swing_high(df_m, janela=OTIMIZADO_POSITION['swing_window'])
            st_sw = st_sw if st_sw and st_sw > entrada else None
            res_ltb = calcular_ltb_adaptativo(df_m)
            st_ltb = res_ltb[0] if res_ltb else None
            if res_ltb and LOG_FILTROS_DETALHADO: logger.log(f"📐 {ticker} LTB: janela={res_ltb[1]}", "DEBUG")
            for met, sl in [('ATR', st_atr), ('Swing High', st_sw), ('LTB Pivôs', st_ltb)]:
                if sl is None or sl <= entrada: continue
                r = sl - entrada
                al = entrada - (r * 3)
                if al >= rl * 0.90:
                    setups.append({'Ticker': ticker, 'Modalidade': 'Position', 'Direcao': 'VENDA', 'Entrada': round(entrada,2), 'Método Stop': met, 'Stop Loss': round(sl,2), 'Risco (R$)': round(r,2), 'Alvo 3:1': round(al,2), 'Resistência': round(rh,2), 'Suporte': round(rl,2), 'Regime': reg, 'Eficiência': ef, 'Sentimento': sentimento, 'Volume Anormal': vol_an, 'Alvos Fibonacci': fib_v})
        return setups if setups else None
    except Exception as e:
        if LOG_FILTROS_DETALHADO: logger.log(f"Erro position {ticker}: {str(e)[:100]}", "ERRO")
        return None

logger.log("✅ Funções de análise carregadas", "INFO")
print("✅ Célula 3 carregada.")

In [ ]:
# =============================================================================
# CÉLULA 4: EXECUÇÃO PRINCIPAL (v5.2 – integração completa)
# =============================================================================

try:
    from google.colab import userdata
    if not EMAIL_REMETENTE:
        EMAIL_REMETENTE = userdata.get('TRADING_EMAIL')
    if not SENHA_APP:
        SENHA_APP = userdata.get('GMAIL_APP_PASSWORD')
except:
    pass

VOLUME_MINIMO_ACAO = 1_000_000
VOLUME_FINANCEIRO_MINIMO = 1_000_000
LIMITE_LIQUIDEZ_FINANCEIRA = 5_000_000
EXIGIR_CONFLUENCIA = True

def montar_tabela_html(oportunidades, titulo, regime_vol, nh_nl=None):
    if not oportunidades: return ""
    corpo = f"<h3>{titulo}</h3><table border='1' cellpadding='4' cellspacing='0' style='border-collapse:collapse;'>"
    corpo += "<tr><th>Ticker</th><th>Dir.</th><th>Entrada</th><th>Stop</th><th>Alvo Rec.</th><th>Método</th><th>Payoff Real</th><th>Vol. Anormal</th><th>Armadilha</th><th>Padrões</th><th>RSI</th><th>Boll.</th><th>Dow</th><th>Elliott</th><th>Lote</th><th>Score</th><th>Rev.</th></tr>"
    for op in oportunidades:
        vol_an_icon = '🔥' if op.get('Volume Anormal') else '✅'
        armadilha_icon = '🪤' if op.get('Is Armadilha') else '—'
        padroes = op.get('Padrões Detectados', '—')
        rsi = op.get('RSI', '—')
        boll = op.get('Bollinger_Pos', '—')
        dow = op.get('Tendencia_Dow', '—')
        elliott = op.get('Onda_Elliott', '—')
        revisar = '🔍' if op.get('Requer Revisao') else '✅'
        corpo += f"<tr><td>{op['Ticker']}</td><td>{op['Direcao']}</td><td>R$ {op['Entrada']:.2f}</td><td>R$ {op['Stop Loss']:.2f}</td><td>R$ {op['Alvo Recomendado']:.2f}</td><td>{op['Método Alvo']}</td><td>{op['Payoff Real']}:1</td><td>{vol_an_icon}</td><td>{armadilha_icon}</td><td>{padroes}</td><td>{rsi}</td><td>{boll}%</td><td>{dow}</td><td>{elliott}</td><td>{op['Lote']}</td><td>{op.get('Score', 'N/A')}</td><td>{revisar}</td></tr>"
    corpo += f"</table><br><p><small>Custos: {PARAMS_ATIVOS['custos_pct']*100:.1f}% | Regime: {regime_vol} | NH‑NL: {nh_nl}% | v5.2</small></p>"
    return corpo

def enviar_email_ou_exibir(oportunidades, modalidade, regime_vol, nh_nl):
    if not oportunidades:
        logger.log(f"Nenhuma oportunidade de {modalidade} encontrada", "INFO")
        return
    if EMAIL_REMETENTE and SENHA_APP:
        try:
            msg = MIMEMultipart()
            msg['From'] = EMAIL_REMETENTE
            msg['To'] = EMAIL_REMETENTE
            msg['Subject'] = f"🚨 Oportunidades {modalidade} - {datetime.now().strftime('%d/%m/%Y')}"
            msg.attach(MIMEText(montar_tabela_html(oportunidades, "Setups Aprovados", regime_vol, nh_nl), 'html'))
            with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
                server.login(EMAIL_REMETENTE, SENHA_APP)
                server.send_message(msg)
            logger.log(f"E-mail ({modalidade}) enviado", "INFO")
        except Exception as e:
            logger.log(f"Falha no e-mail: {str(e)[:100]}", "ERRO")
    else:
        logger.log(f"E-mail não configurado. Exibindo na tela", "INFO")
    df_op = pd.DataFrame(oportunidades)
    cols = ['Ticker', 'Direcao', 'Entrada', 'Método Stop', 'Stop Loss', 'Risco (R$)', 'Alvo 3:1', 'Alvo Recomendado', 'Payoff Real', 'Resistência', 'Suporte', 'Regime', 'Eficiência', 'Volume Anormal', 'Is Armadilha', 'Padrões Detectados', 'RSI', 'MACD', 'Estocastico_K', 'Bollinger_Pos', 'Climax_Volume', 'Tendencia_Dow', 'Fib_Retracao_382', 'Fib_Retracao_618', 'Onda_Elliott', 'Fase Wyckoff', 'Score', 'Requer Revisao', 'Kelly %', 'Lote']
    try:
        from IPython.display import display
        display(df_op[cols].sort_values(['Direcao', 'Ticker']))
    except:
        print(df_op[cols].sort_values(['Direcao', 'Ticker']).to_string())
    csv_name = f"oportunidades_{modalidade.lower()}_{datetime.now().strftime('%Y%m%d')}.csv"
    df_op[cols].to_csv(csv_name, index=False)
    try:
        from google.colab import files
        files.download(csv_name)
    except:
        logger.log(f"Arquivo '{csv_name}' salvo", "INFO")

def obter_tickers_b3():
    if os.path.exists(CACHE_TICKERS_FILE):
        try:
            with open(CACHE_TICKERS_FILE, 'r', encoding='utf-8') as f:
                cache = json.load(f)
            idade = (datetime.now() - datetime.fromisoformat(cache['timestamp'])).total_seconds() / 3600
            if idade < 24:
                logger.log(f"📦 Usando cache de tickers ({len(cache['tickers'])} ativos)", "INFO")
                return cache['tickers']
        except: pass
    tickers = None
    try:
        url = "https://www.dadosdemercado.com.br/acoes"
        soup = BeautifulSoup(requests.get(url, timeout=10).content, 'html.parser')
        tickers = [row.find_all('td')[0].text.strip() for row in soup.select('table tbody tr') if row.find_all('td') and not row.find_all('td')[0].text.strip().startswith('#')]
        if tickers:
            with open(CACHE_TICKERS_FILE, 'w', encoding='utf-8') as f:
                json.dump({'timestamp': datetime.now().isoformat(), 'tickers': tickers}, f, ensure_ascii=False)
            logger.log(f"🌐 Tickers obtidos via scraping ({len(tickers)} ativos)", "INFO")
            return tickers
    except Exception as e:
        logger.log(f"⚠️ Scraping falhou: {str(e)[:80]}", "WARN")
    logger.log("🔄 Usando fallback de tickers", "WARN")
    return FALLBACK_TICKERS.copy()

# ============================
# EXECUÇÃO PRINCIPAL
# ============================
logger.iniciar_etapa("Coleta de Tickers")
tickers_b3 = obter_tickers_b3()
tickers_b3 = [t.replace('.SA', '') for t in tickers_b3]
logger.concluir_etapa("Coleta de Tickers", {'total': len(tickers_b3)})

tickers_yahoo = [t + ".SA" for t in tickers_b3]
tickers_liquidos = []
BATCH = 50

logger.iniciar_etapa("Filtro de Liquidez")
for i in range(0, len(tickers_yahoo), BATCH):
    batch = tickers_yahoo[i:i+BATCH]
    try:
        data = yf.download(batch, period='3mo', interval='1d', group_by='ticker', progress=False, auto_adjust=True)
        for t in batch:
            if t in TICKERS_BLOQUEADOS: continue
            try:
                if t not in data: continue
                df = data[t].copy()
                if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.droplevel(1)
                df.columns = [c.lower() for c in df.columns]
                df.rename(columns={'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}, inplace=True)
                if 'Volume' not in df.columns or df.empty: continue
                vol = df['Volume'].rolling(21).mean().iloc[-1]
                px = df['Close'].iloc[-1]
                if pd.notna(vol) and pd.notna(px) and vol >= VOLUME_MINIMO_ACAO and (vol * px) >= VOLUME_FINANCEIRO_MINIMO:
                    tickers_liquidos.append(t)
            except: continue
    except Exception as e:
        logger.log(f"Erro no lote {i//BATCH}: {str(e)[:100]}", "ERRO")
    time.sleep(1)

if len(tickers_liquidos) < 10:
    logger.log("Poucos ativos líquidos, usando fallback", "WARN")
    tickers_liquidos = [t + ".SA" for t in FALLBACK_TICKERS]
logger.concluir_etapa("Filtro de Liquidez", {'liquidos': len(tickers_liquidos)})

logger.iniciar_etapa("Download de Dados (5 anos)")
data_d = {}
falhas = []
try:
    data_d_raw = yf.download(tickers_liquidos, period='5y', interval='1d', group_by='ticker', progress=False, auto_adjust=True)
    for t in tickers_liquidos:
        try:
            if t in data_d_raw:
                df_t = data_d_raw[t].copy()
                if isinstance(df_t.columns, pd.MultiIndex): df_t.columns = df_t.columns.droplevel(1)
                df_t.columns = [col.lower() for col in df_t.columns]
                df_t.rename(columns={'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}, inplace=True)
                data_d[t] = df_t
            else: falhas.append(t)
        except: falhas.append(t)
except Exception as e:
    logger.log(f"Yahoo Finance falhou: {str(e)[:100]}", "ERRO")
    falhas = tickers_liquidos.copy()
if falhas: logger.log(f"⚠️ {len(falhas)} tickers falharam", "WARN")
time.sleep(1)
logger.concluir_etapa("Download de Dados", {'sucesso': len(data_d), 'falhas': len(falhas)})

def resample_tf(df, freq, min_days=4, min_days_monthly=10):
    if df is None or df.empty: return None
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex): df.index = pd.to_datetime(df.index)
    agg = {'Open':'first', 'High':'max', 'Low':'min', 'Close':'last', 'Volume':'sum'}
    df_r = df.resample(freq, closed='left', label='left').agg(agg)
    if freq.startswith('W'):
        counts = df.resample(freq, closed='left', label='left').count()['Close']
        df_r = df_r[counts >= min_days]
    elif freq in ('ME', 'M'):
        counts = df.resample(freq, closed='left', label='left').count()['Close']
        df_r = df_r[counts >= min_days_monthly]
    return df_r.dropna()

logger.iniciar_etapa("Resample Semanal/Mensal")
data_w, data_m = {}, {}
for t in tickers_liquidos:
    try:
        if t in data_d and not data_d[t].empty:
            df_d = data_d[t].copy()
            if isinstance(df_d.columns, pd.MultiIndex): df_d.columns = df_d.columns.droplevel(1)
            data_w[t] = resample_tf(df_d, 'W-FRI')
            data_m[t] = resample_tf(df_d, 'ME', min_days_monthly=10)
    except: continue
logger.concluir_etapa("Resample", {'semanais': len(data_w), 'mensais': len(data_m)})

# NH‑NL (sentimento de mercado)
nh_nl = calcular_nh_nl_simplificado(tickers_liquidos, data_w)
logger.log(f"📊 NH‑NL: {nh_nl}% dos ativos acima da MM50 semanal", "INFO")

logger.iniciar_etapa("Regime de Volatilidade")
ibov = None
for simbolo in ["^IBOV", "^BVSP", "BOVA11.SA"]:
    try:
        ibov_raw = yf.download(simbolo, period='3mo', interval='1d', progress=False)
        if not ibov_raw.empty and 'Close' in ibov_raw.columns:
            ibov = ibov_raw['Close'].dropna()
            if isinstance(ibov, pd.DataFrame): ibov = ibov.squeeze()
            if len(ibov) >= 60:
                logger.log(f"✅ IBOV via {simbolo}", "INFO")
                break
    except: pass

if ibov is not None and len(ibov) >= 60:
    regime_vol = detectar_regime_volatilidade(ibov)
    PARAMS_ATIVOS.clear()
    PARAMS_ATIVOS.update(PARAMS_ALTA_VOL if regime_vol == 'ALTA' else PARAMS_BAIXA_VOL)
else:
    regime_vol = 'BAIXA'
    PARAMS_ATIVOS.clear()
    PARAMS_ATIVOS.update(PARAMS_BAIXA_VOL)
logger.concluir_etapa("Regime", {'regime': regime_vol})

kelly_pct = fractional_kelly(WIN_RATE_ESTIMADO, PAYOFF_ESTIMADO, PARAMS_ATIVOS['kelly_frac'])
risco_maximo = CAPITAL_TOTAL * kelly_pct

def verificar_circuit_breakers():
    if not os.path.exists(ARQUIVO_LOG): return True, None
    try:
        with open(ARQUIVO_LOG, 'r', encoding='utf-8') as f: logs = json.load(f)
        hoje = datetime.now().date()
        trades = [l for l in logs if l['tipo'] == 'TRADE_FECHADO' and datetime.fromisoformat(l['timestamp']).date() == hoje]
        if not trades: return True, None
        pnl = sum(t['dados'].get('pnl_real', 0) for t in trades)
        if abs(pnl) / CAPITAL_TOTAL >= DRAWDOWN_MAX_DIARIO: return False, f"Drawdown >= {DRAWDOWN_MAX_DIARIO*100}%"
        perdas = 0
        for t in reversed(trades):
            if t['dados'].get('pnl_real', 0) < 0: perdas += 1
            else: break
        if perdas >= MAX_PERDAS_CONSECUTIVAS: return False, f"{perdas} perdas seguidas"
        return True, None
    except: return True, None

pode, motivo = verificar_circuit_breakers()
if not pode:
    logger.log(f"🛑 CIRCUIT BREAKER: {motivo}", "ALERT")
    raise SystemExit

oportunidades_swing = []
oportunidades_position = []

def get_df(data, ticker):
    if data and isinstance(data, dict) and ticker in data:
        try:
            df = data[ticker].copy()
            if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.droplevel(1)
            df.rename(columns={'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}, inplace=True)
            return df
        except: pass
    return None

stats_filtros = {'total_analisados': 0, 'passou_preco': 0, 'passou_risco': 0, 'passou_confluencia': 0, 'passou_mm': 0, 'passou_macro': 0, 'passou_wyckoff': 0, 'passou_payoff': 0, 'passou_gap': 0, 'setup_aprovado_swing': 0, 'setup_aprovado_position': 0, 'armadilhas_detectadas': 0, 'padroes_detectados': 0}

preco_minimo = PARAMS_ATIVOS.get('preco_minimo', PRECO_MINIMO)
risco_max_pct = PARAMS_ATIVOS.get('risco_percentual_maximo', RISCO_PERCENTUAL_MAXIMO)
gap_max_pct = PARAMS_ATIVOS['gap_max_pct']

logger.iniciar_etapa("Análise de Setups")

for i, ticker in enumerate(tickers_liquidos):
    if LOG_FILTROS_DETALHADO and i % 20 == 0:
        logger.log(f"Progresso: {i+1}/{len(tickers_liquidos)}", "DEBUG")
    df_w = get_df(data_w, ticker)
    df_d_local = get_df(data_d, ticker)
    df_m_local = get_df(data_m, ticker)
    if df_w is None or df_w.empty: continue
    stats_filtros['total_analisados'] += 1
    vol_fin = None
    try:
        vfc = (df_w['Volume'] * df_w['Close']).rolling(20).mean()
        vol_fin = vfc.iloc[-1] if pd.notna(vfc.iloc[-1]) else None
    except: pass

    # ================= SWING =================
    res_swing = analisar_swing_trade(ticker, df_w=df_w, df_d=df_d_local)
    if res_swing:
        for r in res_swing:
            e = r['Entrada']
            rp = r['Risco (R$)'] / e
            if e < preco_minimo: continue
            stats_filtros['passou_preco'] += 1
            if rp < RISCO_PERCENTUAL_MINIMO or rp > risco_max_pct: continue
            stats_filtros['passou_risco'] += 1
            if EXIGIR_CONFLUENCIA and (r['Regime'] not in [1,2] or r['Eficiência'] is None or r['Eficiência'] < 0.6): continue
            stats_filtros['passou_confluencia'] += 1
            mm200w = df_w['Close'].rolling(200).mean().iloc[-1]
            if pd.notna(mm200w):
                if (r['Direcao'] == 'COMPRA' and e < mm200w) or (r['Direcao'] == 'VENDA' and e > mm200w): continue
            if df_d_local is not None and not df_d_local.empty:
                mm200d = df_d_local['Close'].rolling(200).mean().iloc[-1]
                if pd.notna(mm200d):
                    if (r['Direcao'] == 'COMPRA' and e < mm200d) or (r['Direcao'] == 'VENDA' and e > mm200d): continue
            stats_filtros['passou_mm'] += 1
            if PARAMS_ATIVOS.get('exigir_volume_anormal', False) and not r.get('Volume Anormal', True): continue
            alinhado_macro, _ = verificar_alinhamento_macro(ticker, r['Direcao'])
            if not alinhado_macro: continue
            stats_filtros['passou_macro'] += 1
            q_vol, s_vol = avaliar_qualidade_volume(df_w, r['Direcao'])
            wyck, w_conf = detectar_fase_wyckoff_adaptativo(df_w, r.get('Suporte'), r.get('Resistência'))
            if (r['Direcao'] == 'COMPRA' and wyck == 'DISTRIBUICAO') or (r['Direcao'] == 'VENDA' and wyck == 'ACUMULACAO'): continue
            stats_filtros['passou_wyckoff'] += 1

            # ---- NOVOS PADRÕES (v5.1) ----
            padroes = []
            pc = analisar_candle(df_w.iloc[-1])
            for nome, val in pc.items():
                if val: padroes.append(nome)
            oco = detectar_oco(df_w)
            if oco: padroes.append(oco['tipo'])
            band = detectar_bandeira(df_w)
            if band: padroes.append(band['tipo'])
            tri_asc = detectar_triangulo_ascendente(df_w)
            if tri_asc: padroes.append('Triang_asc')
            tri_desc = detectar_triangulo_descendente(df_w)
            if tri_desc: padroes.append('Triang_desc')
            gap_tipo = classificar_gap(df_w, r['Direcao'])
            if gap_tipo: padroes.append(gap_tipo['tipo'])
            if padroes: stats_filtros['padroes_detectados'] += len(padroes)

            # ---- NOVOS MACRO + TÉCNICOS (v5.2) ----
            rsi_val = calcular_rsi(df_w)
            macd_val, macd_sinal, macd_hist = calcular_macd(df_w)
            stoch_k, stoch_d = calcular_estocastico(df_w)
            bb = calcular_bandas_bollinger(df_w)
            climax = calcular_climax_volume(df_w)
            dow = detectar_estrutura_dow(df_w)
            fib_ret = calcular_fibonacci_retracao(df_w)
            elliott = classificar_onda_elliott(df_w)

            is_arm, forca_arm = False, 0.0
            if r['Direcao'] == 'COMPRA': is_arm, forca_arm = detectar_armadilha_lta(df_w, r.get('Suporte', e), BANDA_ZONA_PCT)
            else: is_arm, forca_arm = detectar_armadilha_ltb(df_w, r.get('Resistência', e), BANDA_ZONA_PCT)
            if is_arm: stats_filtros['armadilhas_detectadas'] += 1

            mult = calcular_score_confianca(r, alinhado_macro, q_vol, s_vol, wyck, w_conf, is_arm, forca_arm)
            fat_liq = min(1.0, vol_fin / LIMITE_LIQUIDEZ_FINANCEIRA) if pd.notna(vol_fin) else 0.5
            lote_base = int(risco_maximo / r['Risco (R$)'])
            lote_aj = int(lote_base * mult * fat_liq)
            if lote_aj == 0: continue
            al_rec, met_al = calcular_alvo_recomendado(r)
            p_real = calcular_payoff_real(e, al_rec, r['Stop Loss'], PARAMS_ATIVOS['custos_pct'])
            if p_real < 2.0: continue
            stats_filtros['passou_payoff'] += 1
            if len(df_w) >= 2:
                gap = abs(e - df_w['Close'].iloc[-2]) / df_w['Close'].iloc[-2]
                if gap > gap_max_pct: continue
            stats_filtros['passou_gap'] += 1
            score_int = int(mult * 100)
            requer_revisao = (wyck == 'INDEFINIDO' or score_int < 50)
            r.update({'Alvo Recomendado': round(al_rec,2), 'Método Alvo': met_al, 'Payoff Real': p_real, 'Kelly %': round((kelly_pct*mult*fat_liq)*100,2), 'Lote': lote_aj, 'Score': score_int, 'Fase Wyckoff': wyck, 'Wyckoff Conf': round(w_conf,2), 'Regime Vol': regime_vol, 'Is Armadilha': is_arm, 'Forca Armadilha': round(forca_arm,2), 'Requer Revisao': requer_revisao, 'Padrões Detectados': ', '.join(padroes) if padroes else 'Nenhum',
                       'RSI': rsi_val, 'MACD': f"{macd_val}/{macd_sinal}/{macd_hist}" if macd_val else None, 'Estocastico_K': stoch_k, 'Estocastico_D': stoch_d, 'Bollinger_Pos': bb['posicao_%'] if bb else None, 'Climax_Volume': climax, 'Tendencia_Dow': dow['tendencia_dow'] if dow else None, 'Fib_Retracao_382': fib_ret['38.2%'] if fib_ret else None, 'Fib_Retracao_618': fib_ret['61.8%'] if fib_ret else None, 'Onda_Elliott': elliott, 'NH_NL': nh_nl})
            if HABILITAR_LOGGING: log_evento('SETUP', ticker, {'mod': 'Swing', 'score': score_int, 'padroes': padroes, 'rsi': rsi_val})
            stats_filtros['setup_aprovado_swing'] += 1
            oportunidades_swing.append(r)

    # ================= POSITION =================
    if df_m_local is not None and not df_m_local.empty and df_w is not None:
        res_pos = analisar_position_trade(ticker, df_m=df_m_local, df_w=df_w)
        if res_pos:
            for r in res_pos:
                e = r['Entrada']
                rp = r['Risco (R$)'] / e
                if e < preco_minimo: continue
                if rp < 0.02 or rp > 0.40: continue
                if EXIGIR_CONFLUENCIA and (r['Regime'] not in [1,2] or r['Eficiência'] is None or r['Eficiência'] < 0.5): continue
                mm50m = df_m_local['Close'].rolling(50).mean().iloc[-1]
                if pd.notna(mm50m):
                    if (r['Direcao'] == 'COMPRA' and e < mm50m) or (r['Direcao'] == 'VENDA' and e > mm50m): continue
                alinhado_macro, _ = verificar_alinhamento_macro(ticker, r['Direcao'])
                if not alinhado_macro: continue
                q_vol, s_vol = avaliar_qualidade_volume(df_m_local, r['Direcao'])
                wyck, w_conf = detectar_fase_wyckoff_adaptativo(df_m_local, r.get('Suporte'), r.get('Resistência'))
                if (r['Direcao'] == 'COMPRA' and wyck == 'DISTRIBUICAO') or (r['Direcao'] == 'VENDA' and wyck == 'ACUMULACAO'): continue

                # Padrões + Macro/Técnico
                padroes = []
                pc = analisar_candle(df_m_local.iloc[-1])
                for nome, val in pc.items():
                    if val: padroes.append(nome)
                oco = detectar_oco(df_m_local)
                if oco: padroes.append(oco['tipo'])
                band = detectar_bandeira(df_m_local)
                if band: padroes.append(band['tipo'])
                tri_asc = detectar_triangulo_ascendente(df_m_local)
                if tri_asc: padroes.append('Triang_asc')
                tri_desc = detectar_triangulo_descendente(df_m_local)
                if tri_desc: padroes.append('Triang_desc')
                gap_tipo = classificar_gap(df_m_local, r['Direcao'])
                if gap_tipo: padroes.append(gap_tipo['tipo'])
                if padroes: stats_filtros['padroes_detectados'] += len(padroes)

                rsi_val = calcular_rsi(df_m_local)
                macd_val, macd_sinal, macd_hist = calcular_macd(df_m_local)
                stoch_k, stoch_d = calcular_estocastico(df_m_local)
                bb = calcular_bandas_bollinger(df_m_local)
                climax = calcular_climax_volume(df_m_local)
                dow = detectar_estrutura_dow(df_m_local)
                fib_ret = calcular_fibonacci_retracao(df_m_local)
                elliott = classificar_onda_elliott(df_m_local)

                is_arm, forca_arm = False, 0.0
                if r['Direcao'] == 'COMPRA': is_arm, forca_arm = detectar_armadilha_lta(df_m_local, r.get('Suporte', e), BANDA_ZONA_PCT)
                else: is_arm, forca_arm = detectar_armadilha_ltb(df_m_local, r.get('Resistência', e), BANDA_ZONA_PCT)
                if is_arm: stats_filtros['armadilhas_detectadas'] += 1

                mult = calcular_score_confianca(r, alinhado_macro, q_vol, s_vol, wyck, w_conf, is_arm, forca_arm)
                vol_fin_m = None
                try:
                    vfmc = (df_m_local['Volume'] * df_m_local['Close']).rolling(20).mean()
                    vol_fin_m = vfmc.iloc[-1] if pd.notna(vfmc.iloc[-1]) else None
                except: pass
                fat_liq_m = min(1.0, vol_fin_m / LIMITE_LIQUIDEZ_FINANCEIRA) if pd.notna(vol_fin_m) else 0.5
                lote_base = int(risco_maximo / r['Risco (R$)'])
                lote_aj = int(lote_base * mult * fat_liq_m)
                if lote_aj == 0: continue
                al_rec, met_al = calcular_alvo_recomendado(r)
                p_real = calcular_payoff_real(e, al_rec, r['Stop Loss'], PARAMS_ATIVOS['custos_pct'])
                if p_real < 2.0: continue
                if len(df_m_local) >= 2:
                    gap = abs(e - df_m_local['Close'].iloc[-2]) / df_m_local['Close'].iloc[-2]
                    if gap > gap_max_pct: continue
                score_int = int(mult * 100)
                requer_revisao = (wyck == 'INDEFINIDO' or score_int < 50)
                r.update({'Alvo Recomendado': round(al_rec,2), 'Método Alvo': met_al, 'Payoff Real': p_real, 'Kelly %': round((kelly_pct*mult*fat_liq_m)*100,2), 'Lote': lote_aj, 'Score': score_int, 'Fase Wyckoff': wyck, 'Wyckoff Conf': round(w_conf,2), 'Regime Vol': regime_vol, 'Is Armadilha': is_arm, 'Forca Armadilha': round(forca_arm,2), 'Requer Revisao': requer_revisao, 'Padrões Detectados': ', '.join(padroes) if padroes else 'Nenhum',
                           'RSI': rsi_val, 'MACD': f"{macd_val}/{macd_sinal}/{macd_hist}" if macd_val else None, 'Estocastico_K': stoch_k, 'Estocastico_D': stoch_d, 'Bollinger_Pos': bb['posicao_%'] if bb else None, 'Climax_Volume': climax, 'Tendencia_Dow': dow['tendencia_dow'] if dow else None, 'Fib_Retracao_382': fib_ret['38.2%'] if fib_ret else None, 'Fib_Retracao_618': fib_ret['61.8%'] if fib_ret else None, 'Onda_Elliott': elliott, 'NH_NL': nh_nl})
                if HABILITAR_LOGGING: log_evento('SETUP', ticker, {'mod': 'Position', 'score': score_int, 'padroes': padroes, 'rsi': rsi_val})
                stats_filtros['setup_aprovado_position'] += 1
                oportunidades_position.append(r)

logger.concluir_etapa("Análise de Setups", stats_filtros)

if len(oportunidades_swing) > MAX_SETUPS_POR_DIA:
    oportunidades_swing = sorted(oportunidades_swing, key=lambda x: x['Score'], reverse=True)[:MAX_SETUPS_POR_DIA]
if len(oportunidades_position) > MAX_SETUPS_POR_DIA:
    oportunidades_position = sorted(oportunidades_position, key=lambda x: x['Score'], reverse=True)[:MAX_SETUPS_POR_DIA]

logger.log(f"\n🎯 Swing: {len(oportunidades_swing)} | Position: {len(oportunidades_position)} setups", "RESULTADO")
logger.log(f"   Kelly: {kelly_pct*100:.2f}% | Regime: {regime_vol} | NH‑NL: {nh_nl}%", "RESULTADO")
logger.log(f"   🪤 Armadilhas: {stats_filtros['armadilhas_detectadas']} | 📐 Padrões: {stats_filtros['padroes_detectados']}", "RESULTADO")

if oportunidades_swing: enviar_email_ou_exibir(oportunidades_swing, "Swing Trade", regime_vol, nh_nl)
if oportunidades_position: enviar_email_ou_exibir(oportunidades_position, "Position Trade", regime_vol, nh_nl)

logger.resumo_final()
logger.log("✅ v5.2 concluído", "SUCCESS")
print("\n✅ Execução concluída.")